In [ ]:
%matplotlib inline
import sys, os, json
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from scipy.optimize import brentq, root, least_squares
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

In [ ]:
# ==========================================
# EXPERIMENT SETTINGS
# ==========================================
WORKBOOK = "CfD distillation simulation 1 (2).xlsx"
RESULTS_DIR = "results"   # results workbook and charts are written here

RUN_ALL_TABS = True       # True: every data tab in the workbook (bad tabs are reported and skipped). False: only the tab chosen below.
PRESSURE_BAR = 3          # tab pressure: 3, 2, 1, 0.5 or 0.2
FEED_STAGE = 3            # tab feed stage: 1..5 (1 = top, 5 = boiler)
 
FEED_FRACTION_LOW = 0.30  # mass fraction of the low boiler (n-hexane) in the feed (workbook: 30 %)
TRAY_EFFICIENCY = 1.0     # 0..1. 1.0 = ideal stages (what Aspen assumes). Your 0.75 lowers purity.
 
# How the stage temperature is obtained:
#   "bubble" : calculated from liquid composition and stage pressure (recommended)
#   "excel"  : temperatures taken from column I of the workbook
#   "factor" : your rule, boiler temperature x 0.95 for every stage higher up
TEMPERATURE_MODE = "bubble"
BOILER_TEMPERATURE = 400.0   # K, only used for "factor"
TEMPERATURE_FACTOR = 0.95    # only used for "factor"

In [ ]:
 
# ==========================================
# COMPONENT DATA
# ==========================================
LOW, HIGH = "N-HEX", "EB"                       # low / high boiling component
MW = {LOW: 86.178, HIGH: 106.167}               # g/mol
# Antoine: log10(P[mmHg]) = A - B / (C + T[degC]); typical literature values, check against your source
ANTOINE = {LOW: (6.87776, 1171.17, 224.408),
           HIGH: (6.95719, 1424.255, 213.21)}
MMHG_TO_BAR = 0.00133322
 
STAGE_ROWS = range(4, 9)    # rows 4..8   -> stages 1..5 (flows, T, P)
LIQ_ROWS = range(12, 17)    # rows 12..16 -> liquid composition (EB, N-HEX)
VAP_ROWS = range(20, 25)    # rows 20..24 -> vapour composition (EB, N-HEX)
N_STAGES = len(STAGE_ROWS)  # the column has 5 stages: 1 = top, 5 = boiler
FLOW_COLUMNS = {12: "L", 13: "M", 14: "N", 17: "Q"}   # flow columns the model reads (must be in kg/hr)
NOTES_SHEET = "Blad1"       # the only tab without column data
MASS_BALANCE_TOLERANCE = 0.01   # products may differ from the feed by at most 1 %

In [ ]:
# ==========================================
# READ AND CHECK THE WORKBOOK
# ==========================================
class ColumnInputError(ValueError):
    """The settings or the workbook data are wrong, so the tab must not be run."""


def check_settings():
    """Checks the experiment settings before anything is read."""
    if not os.path.isfile(WORKBOOK):
        raise ColumnInputError(f"Workbook '{WORKBOOK}' was not found in {os.getcwd()}.")
    if not 0 < FEED_FRACTION_LOW < 1:
        raise ColumnInputError(f"FEED_FRACTION_LOW must be between 0 and 1, got {FEED_FRACTION_LOW}.")
    if not 0 < TRAY_EFFICIENCY <= 1:
        raise ColumnInputError(f"TRAY_EFFICIENCY must be above 0 and at most 1, got {TRAY_EFFICIENCY}.")
    if TEMPERATURE_MODE not in ("bubble", "excel", "factor"):
        raise ColumnInputError(f"TEMPERATURE_MODE must be 'bubble', 'excel' or 'factor', got '{TEMPERATURE_MODE}'.")


def check_request(pressure, feed_stage):
    """A pressure and feed stage must be physically possible for this column."""
    if pressure <= 0:
        raise ColumnInputError(f"Pressure must be above 0 bar, got {pressure:g} bar.")
    if float(feed_stage) != int(feed_stage):
        raise ColumnInputError(f"Feed stage must be a whole number, got {feed_stage:g}.")
    if not 1 <= feed_stage <= N_STAGES:
        raise ColumnInputError(f"Feed stage {feed_stage:g} does not exist: "
                               f"the column has {N_STAGES} stages (1 = top, {N_STAGES} = boiler).")


def sheet_name(pressure, feed_stage):
    return f"{pressure:g}".replace(".", ",") + f",{int(feed_stage)}"      # 0.5 -> "0,5,3"


def parse_tab_name(name):
    """Turns a tab name into (pressure, feed stage): "0,5,3" -> (0.5, 3). Spaces are ignored."""
    parts = name.replace(" ", "").split(",")
    try:
        pressure, feed_stage = float(".".join(parts[:-1])), float(parts[-1])
    except ValueError:
        raise ColumnInputError(f"Tab name '{name}' is not in the form 'pressure,feed stage'.") from None
    return pressure, int(feed_stage) if feed_stage.is_integer() else feed_stage


def find_sheet(wb, pressure, feed_stage):
    """Finds the tab of an experiment. Spaces in tab names ("0,8 ,2") are ignored."""
    name = sheet_name(pressure, feed_stage)
    for tab in wb.sheetnames:
        if tab.replace(" ", "") == name:
            return wb[tab]
    raise ColumnInputError(f"There is no tab '{name}' in the workbook.")


def read_number(ws, row, column):
    """Reads one cell. A blank or text cell is an error, it is never treated as 0."""
    cell = ws.cell(row, column)
    if isinstance(cell.value, bool) or not isinstance(cell.value, (int, float)):
        raise ColumnInputError(f"Cell {cell.coordinate} must contain a number, but contains {cell.value!r}.")
    return float(cell.value)


def check_layout(ws):
    """The tab must have the 5-stage layout and kg/hr flows that the model reads."""
    stages = [ws.cell(r, 8).value for r in STAGE_ROWS]
    if stages != list(range(1, N_STAGES + 1)):
        raise ColumnInputError(f"Cells H4:H8 must contain stages 1 to {N_STAGES}, but contain {stages}.")
    next_stage = ws.cell(STAGE_ROWS[-1] + 1, 8).value
    if isinstance(next_stage, (int, float)):
        raise ColumnInputError(f"The tab has more than {N_STAGES} stages (cell H{STAGE_ROWS[-1] + 1} "
                               f"contains stage {next_stage:g}), but the column has only {N_STAGES}.")
    for column, letter in FLOW_COLUMNS.items():
        unit = ws.cell(3, column).value
        if unit != "kg/hr":
            raise ColumnInputError(f"Flows must be in kg/hr, but cell {letter}3 says '{unit}'.")


def check_stage_data(df, pressure, feed_stage):
    """The numbers in the tab must match the tab name and make physical sense."""
    if abs(df["P_bar"].iloc[0] - pressure) > 1e-3:
        raise ColumnInputError(f"The tab name says {pressure:g} bar, but the top stage is at {df['P_bar'].iloc[0]:g} bar.")
    if (df["P_bar"] <= 0).any():
        raise ColumnInputError("Every stage pressure must be above 0 bar.")
    for column in ("L", "V", "feed", "product"):
        if (df[column] < 0).any():
            raise ColumnInputError(f"Column '{column}' contains a negative flow.")
    fed_stages = df.loc[df["feed"] > 0, "stage"].tolist()
    if len(fed_stages) != 1:
        raise ColumnInputError(f"The column must have exactly one feed, but the tab has a feed on stages {fed_stages}.")
    if fed_stages[0] != feed_stage:
        raise ColumnInputError(f"The tab name says feed stage {feed_stage:g}, but the feed is on stage {fed_stages[0]}.")
    if df["product"].iloc[0] <= 0 or df["product"].iloc[-1] <= 0:
        raise ColumnInputError("The column must have a top product (stage 1) and a bottom product "
                               f"(stage {N_STAGES}).")
    feed, products = df["feed"].sum(), df["product"].sum()
    if abs(feed - products) > MASS_BALANCE_TOLERANCE * feed:
        raise ColumnInputError(f"Mass balance does not close: feed {feed:.1f} kg/h, products {products:.1f} kg/h.")
    for phase, high, low in (("liquid", "x_HIGH", "x_LOW"), ("vapour", "y_HIGH", "y_LOW")):
        if ((df[[high, low]] < 0) | (df[[high, low]] > 1)).any(axis=None):
            raise ColumnInputError(f"The {phase} mass fractions must be between 0 and 1.")
        if ((df[high] + df[low] - 1).abs() > 1e-3).any():
            raise ColumnInputError(f"The {phase} mass fractions must add up to 1 on every stage.")


def read_stage_data(ws):
    """Stage table for one tab. Component mass = mass fraction x stream flow (kg/h)."""
    check_layout(ws)
    g = lambda r, c: read_number(ws, r, c)
    df = pd.DataFrame({
        "stage": [int(ws.cell(r, 8).value) for r in STAGE_ROWS],
        "T_excel_K": [g(r, 9) + 273.15 for r in STAGE_ROWS],
        "P_bar": [g(r, 10) for r in STAGE_ROWS],
        "L": [g(r, 12) for r in STAGE_ROWS],          # liquid leaving stage (kg/h)
        "V": [g(r, 13) for r in STAGE_ROWS],          # vapour leaving stage (kg/h)
        "feed": [g(r, 14) for r in STAGE_ROWS],       # liquid feed (kg/h)
        "product": [g(r, 17) for r in STAGE_ROWS],    # liquid product drawn (kg/h)
        "x_HIGH": [g(r, 9) for r in LIQ_ROWS],        # Aspen liquid mass fraction EB
        "x_LOW": [g(r, 10) for r in LIQ_ROWS],
        "y_HIGH": [g(r, 9) for r in VAP_ROWS],        # Aspen vapour mass fraction EB
        "y_LOW": [g(r, 10) for r in VAP_ROWS],
    })
    # your formula: mass = ratio x flow  (e.g. liquid EB stage 1 = I12 * L4)
    df["aspen_liq_EB_kg_h"] = df["x_HIGH"] * df["L"]
    df["aspen_liq_HEX_kg_h"] = df["x_LOW"] * df["L"]
    df["aspen_vap_EB_kg_h"] = df["y_HIGH"] * df["V"]
    df["aspen_vap_HEX_kg_h"] = df["y_LOW"] * df["V"]
    return df

In [ ]:
# ==========================================
# THERMODYNAMICS (boiling point, temperature, pressure)
# ==========================================
def psat_bar(comp, T_K):
    A, B, C = ANTOINE[comp]
    return 10 ** (A - B / (C + (T_K - 273.15))) * MMHG_TO_BAR
 
 
def w_to_x(w_high):
    a, b = w_high / MW[HIGH], (1 - w_high) / MW[LOW]
    return a / (a + b)
 
 
def x_to_w(x_high):
    a, b = x_high * MW[HIGH], (1 - x_high) * MW[LOW]
    return a / (a + b)
 
 
def stage_temperature(j, n, w_high, P_bar, T_excel):
    """Temperature of stage j (0 = top)."""
    if TEMPERATURE_MODE == "excel":
        return T_excel[j]
    if TEMPERATURE_MODE == "factor":
        return BOILER_TEMPERATURE * TEMPERATURE_FACTOR ** (n - 1 - j)
    x = w_to_x(w_high)                       # bubble point: sum(K_i x_i) = 1
    f = lambda T: (x * psat_bar(HIGH, T) + (1 - x) * psat_bar(LOW, T)) / P_bar - 1
    try:
        return brentq(f, 250.0, 600.0)
    except ValueError:
        raise RuntimeError(f"No bubble point between 250 and 600 K on stage {j + 1} at {P_bar:g} bar.") from None
 
 
def equilibrium_vapour(w_high, T_K, P_bar):
    """Vapour mass fraction of the high boiler in equilibrium with the liquid (K = Psat/P)."""
    x = w_to_x(w_high)
    k_hi, k_lo = psat_bar(HIGH, T_K) / P_bar, psat_bar(LOW, T_K) / P_bar
    y = x * k_hi / (x * k_hi + (1 - x) * k_lo)
    return x_to_w(y)

In [ ]:
# ==========================================
# COLUMN FUNCTION
# ==========================================
def run_column(data, feed_fraction_low=FEED_FRACTION_LOW, efficiency=TRAY_EFFICIENCY):
    """
    Solve all stages at once (they are coupled: vapour goes up, liquid comes down).
    Per stage j (component balance on the high boiler, kg/h):
        liquid from stage above + vapour from stage below + feed = liquid out + vapour out
    Vapour leaving a stage: y_j = y_below + E * (y_equilibrium - y_below)
    The liquid/vapour flows (L, V) and product draws come from the workbook.
    """
    n = len(data)
    P, T_ex = data["P_bar"].values, data["T_excel_K"].values
    L, V, F, Q = (data[c].values for c in ("L", "V", "feed", "product"))
    D = L - Q                                  # liquid sent down to the next stage
    z_hi = 1 - feed_fraction_low
 
    def stages(w):
        T, y = np.zeros(n), np.zeros(n)
        for j in range(n - 1, -1, -1):         # boiler (bottom) -> top
            T[j] = stage_temperature(j, n, w[j], P[j], T_ex)
            y_eq = equilibrium_vapour(w[j], T[j], P[j])
            y_below = y[j + 1] if j < n - 1 else y_eq     # boiler = equilibrium stage
            y[j] = y_below + efficiency * (y_eq - y_below)
        return T, y
 
    def residuals(w):
        _, y = stages(w)
        r = np.zeros(n)
        for j in range(n):
            liq_in = D[j - 1] * w[j - 1] if j > 0 else 0.0
            vap_in = V[j + 1] * y[j + 1] if j < n - 1 else 0.0
            r[j] = liq_in + vap_in + F[j] * z_hi - L[j] * w[j] - V[j] * y[j]
        return r
 
    w0 = np.full(n, z_hi)
    sol = root(residuals, w0, method="hybr")
    if not sol.success:
        sol = root(residuals, w0, method="lm")
    if not sol.success:
        raise RuntimeError("Column did not converge: " + sol.message)
    w = sol.x
    if np.any(w < -1e-6) or np.any(w > 1 + 1e-6):
        raise RuntimeError(f"The solution is not physical: mass fractions outside 0..1 ({np.round(w, 3).tolist()}).")
    T, y = stages(w)
 
    out = data[["stage", "P_bar"]].copy()
    out["T_K"] = np.round(T, 1)
    out["liq_purity_LOW_%"] = np.round(100 * (1 - w), 2)
    out["liq_purity_HIGH_%"] = np.round(100 * w, 2)
    out["vap_purity_LOW_%"] = np.round(100 * (1 - y), 2)
    out["liq_LOW_kg_h"] = np.round((1 - w) * L).astype(int)
    out["liq_HIGH_kg_h"] = np.round(w * L).astype(int)
    out["vap_LOW_kg_h"] = np.round((1 - y) * V).astype(int)
    out["vap_HIGH_kg_h"] = np.round(y * V).astype(int)
    return out, w

In [ ]:
# ==========================================
# FINAL PRODUCT CALCULATIONS
# ==========================================
def calculate_products(top_low, top_high, bottom_low, bottom_high):
    top_total = top_low + top_high
    bottom_total = bottom_low + bottom_high
    return {
        "top_amount_kg_h": int(round(top_total)),
        "top_purity_N-HEX_%": float(round(100 * top_low / top_total, 2)) if top_total > 0 else 0,
        "bottom_amount_kg_h": int(round(bottom_total)),
        "bottom_purity_EB_%": float(round(100 * bottom_high / bottom_total, 2)) if bottom_total > 0 else 0,
    }
 
 
def products_from_model(data, w):
    top_amt, bot_amt = data["product"].iloc[0], data["product"].iloc[-1]   # stage 1 and stage 5 draws
    return calculate_products(top_amt * (1 - w[0]), top_amt * w[0],
                              bot_amt * (1 - w[-1]), bot_amt * w[-1])
 
 
def products_from_aspen(data):
    top_amt, bot_amt = data["product"].iloc[0], data["product"].iloc[-1]
    x = data["x_HIGH"].values
    return calculate_products(top_amt * (1 - x[0]), top_amt * x[0],
                              bot_amt * (1 - x[-1]), bot_amt * x[-1])
 
 
# ==========================================
# RUN
# ==========================================
def run_experiment(wb, pressure, feed_stage):
    check_request(pressure, feed_stage)
    data = read_stage_data(find_sheet(wb, pressure, feed_stage))
    check_stage_data(data, pressure, feed_stage)
    table, w = run_column(data)
    return data, table, w
 
 
# Chart settings per liquid: name on the chart, Aspen column, digital twin column
CHART_LIQUIDS = {LOW: ("n-hexane", "x_LOW", "liq_purity_LOW_%"),
                 HIGH: ("Ethylbenzene", "x_HIGH", "liq_purity_HIGH_%")}
 
 
def draw_profile(ax, data, table, name, pressure, feed_stage, component):
    """Draws the liquid purity per stage of one component."""
    label, aspen_col, model_col = CHART_LIQUIDS[component]
    ax.set_facecolor("#fcfcfb")
    ax.plot(data["stage"], 100 * data[aspen_col], "o--", color="#2a78d6", lw=2, ms=7, label="Aspen (workbook)")
    ax.plot(table["stage"], table[model_col], "s-", color="#eb6834", lw=2, ms=7, label="Digital twin")
    ax.set_xlabel("Stage (1 = top, 5 = boiler)", color="#52514e")
    ax.set_ylabel(f"{label} in liquid (mass %)", color="#52514e")
    ax.set_title(f"{label} liquid purity, tab {name} ({pressure:g} bar, feed on stage {feed_stage})",
                 color="#0b0b0b", fontsize=11)
    ax.set_xticks(list(data["stage"]))
    ax.set_ylim(0, 100)                 # same scale on every chart, so tabs can be compared
    ax.tick_params(colors="#52514e")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False); ax.grid(alpha=0.3)
 
 
def plot_tab(data, table, name, pressure, feed_stage):
    """Saves one PNG per liquid in results/charts and shows both side by side in the notebook."""
    for component in CHART_LIQUIDS:
        fig, ax = plt.subplots(figsize=(6, 4), facecolor="#fcfcfb")
        draw_profile(ax, data, table, name, pressure, feed_stage, component)
        fig.tight_layout()
        fig.savefig(os.path.join(RESULTS_DIR, "charts", f"profile_{name.replace(',', '_')}_{component}.png"), dpi=150)
        plt.close(fig)
 
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor="#fcfcfb")
    for ax, component in zip(axes, CHART_LIQUIDS):
        draw_profile(ax, data, table, name, pressure, feed_stage, component)
    fig.tight_layout()
    plt.show()
 
 
def stage_results(data, table, name, pressure, feed_stage):
    """Stage table of one tab, with the Aspen values next to the digital twin."""
    out = table.copy()
    out.insert(0, "tab", name)
    out.insert(1, "pressure_bar", pressure)
    out.insert(2, "feed_stage", feed_stage)
    out = out.drop(columns="P_bar")
    out["aspen_T_K"] = np.round(data["T_excel_K"], 1)
    out["aspen_liq_purity_LOW_%"] = np.round(100 * data["x_LOW"], 2)
    out["aspen_liq_purity_HIGH_%"] = np.round(100 * data["x_HIGH"], 2)
    return out
 
 
def run_tab(wb, pressure, feed_stage):
    """Runs one tab, shows its stage table, products and charts, and returns its results."""
    name = sheet_name(pressure, feed_stage)
    data, table, w = run_experiment(wb, pressure, feed_stage)
    m, a = products_from_model(data, w), products_from_aspen(data)
 
    print(f"\n===== TAB {name} | {pressure:g} bar | feed on stage {feed_stage} =====")
    print(table.to_string(index=False))
    print("\nDIGITAL TWIN :", m)
    print("ASPEN (sheet):", a)
    plot_tab(data, table, name, pressure, feed_stage)
 
    row = {"tab": name, "pressure_bar": pressure, "feed_stage": feed_stage,
           "top_amount": m["top_amount_kg_h"],
           "top_N-HEX_%_model": m["top_purity_N-HEX_%"],
           "top_N-HEX_%_aspen": a["top_purity_N-HEX_%"],
           "bottom_amount": m["bottom_amount_kg_h"],
           "bottom_EB_%_model": m["bottom_purity_EB_%"],
           "bottom_EB_%_aspen": a["bottom_purity_EB_%"]}
    return row, stage_results(data, table, name, pressure, feed_stage)
 
 
def run_all_tabs(wb):
    """Runs every data tab. A bad tab is reported and skipped, so the other tabs still give results."""
    results, rejected = [], []
    for tab in wb.sheetnames:
        if tab == NOTES_SHEET:
            continue
        try:
            pressure, feed_stage = parse_tab_name(tab)
            results.append(run_tab(wb, pressure, feed_stage))
        except (ColumnInputError, RuntimeError) as error:
            print(f"\n===== TAB {tab} REJECTED =====\n{error}")
            rejected.append({"tab": tab, "reason": str(error)})
    return results, rejected
 
 
if __name__ == "__main__":
    check_settings()
    wb = load_workbook(WORKBOOK, data_only=True)
    os.makedirs(os.path.join(RESULTS_DIR, "charts"), exist_ok=True)
    print(f"Temperature mode: {TEMPERATURE_MODE} | tray efficiency: {TRAY_EFFICIENCY}")
 
    if RUN_ALL_TABS:
        results, rejected = run_all_tabs(wb)
        if rejected:
            print(f"\n===== {len(rejected)} REJECTED TABS (not in the results) =====")
            print(pd.DataFrame(rejected).to_string(index=False))
        if not results:
            raise ColumnInputError("None of the tabs could be run, so there are no results to save.")
 
        summary = pd.DataFrame([row for row, _ in results])
        stages = pd.concat([stage for _, stage in results], ignore_index=True)
        print("\n===== SUMMARY OF ALL TABS =====")
        print(summary.to_string(index=False))
        with pd.ExcelWriter(os.path.join(RESULTS_DIR, "results_all_tabs.xlsx")) as writer:
            summary.to_excel(writer, sheet_name="summary", index=False)
            stages.to_excel(writer, sheet_name="stages", index=False)
            if rejected:
                pd.DataFrame(rejected).to_excel(writer, sheet_name="rejected_tabs", index=False)
        print(f"\nSaved {RESULTS_DIR}/results_all_tabs.xlsx and {2 * len(summary)} charts in {RESULTS_DIR}/charts")
    else:
        run_tab(wb, PRESSURE_BAR, FEED_STAGE)   # a bad tab stops here with a clear error

In [ ]:
# ==========================================
# OVERVIEW OF ALL TABS (one page per liquid)
# ==========================================
def plot_overview(stages, component):
    """All tabs in one grid (pressure in rows, feed stage in columns), saved as results/overview_<component>.png."""
    label = CHART_LIQUIDS[component][0]
    side = "LOW" if component == LOW else "HIGH"
    pressures = stages["pressure_bar"].unique()
    feed_stages = stages["feed_stage"].unique()
 
    fig, axes = plt.subplots(len(pressures), len(feed_stages), figsize=(14, 12),
                             sharex=True, sharey=True, facecolor="#fcfcfb", squeeze=False)
    for i, p in enumerate(pressures):
        for j, fs in enumerate(feed_stages):
            tab = stages[(stages["pressure_bar"] == p) & (stages["feed_stage"] == fs)]
            ax = axes[i, j]
            ax.set_facecolor("#fcfcfb")
            ax.plot(tab["stage"], tab[f"aspen_liq_purity_{side}_%"], "o--", color="#2a78d6", lw=1.5, ms=5, label="Aspen (workbook)")
            ax.plot(tab["stage"], tab[f"liq_purity_{side}_%"], "s-", color="#eb6834", lw=1.5, ms=5, label="Digital twin")
            ax.set_xticks(list(tab["stage"]))
            ax.set_ylim(0, 100)
            ax.tick_params(colors="#52514e")
            ax.spines[["top", "right"]].set_visible(False)
            ax.grid(alpha=0.3)
            if i == 0:
                ax.set_title(f"Feed on stage {fs}", color="#0b0b0b", fontsize=11)
            if j == 0:
                ax.set_ylabel(f"{p:g} bar", color="#0b0b0b", fontsize=11)
 
    fig.suptitle(f"{label} liquid purity, all tabs", color="#0b0b0b", fontsize=14)
    fig.supxlabel("Stage (1 = top, 5 = boiler)", color="#52514e")
    fig.supylabel(f"{label} in liquid (mass %)", color="#52514e")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right", frameon=False, ncols=2)
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS_DIR, f"overview_{component}.png"), dpi=150)
    plt.show()
 
 
if RUN_ALL_TABS:
    for component in CHART_LIQUIDS:
        plot_overview(stages, component)
    print(f"Saved the overview charts in {RESULTS_DIR}")
else:
    print("The overview needs all tabs: set RUN_ALL_TABS = True and run the notebook again.")